# A wrong role mode is refused where it is declared

A role states its lifecycle mode once, with `@role_mode`. The four members are `ALIVE`, `DEPRECATED`, `SILENCED` and `UNUSED`, and nothing else is accepted: a string that happens to carry the same value is refused at the declaration, not later at assembly.

[Script](17_a_wrong_role_mode.py)


## A member of the enum, and nothing else


In [1]:
"""A role's mode is declared once, and a wrong one is refused where it is written."""

from __future__ import annotations

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.exceptions import RoleSpecTypeError
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.role_mode import RoleMode, role_mode


@role_mode(RoleMode.ALIVE)
class ViewerRole(ApplicationRole):
    """Read the catalog."""

    name = "viewer"
    description = "Can read the catalog"


print("declared:", ViewerRole._role_mode_info)


## The string the enum carries is not the member

`RoleMode.ALIVE.value` is `"alive"`, and a string is not a member of the set. The class statement fails while the module is imported; the name is never bound.


In [2]:
try:

    @role_mode("alive")
    class MistypedRole(ApplicationRole):
        """Written the way the value looks."""

        name = "mistyped"
        description = "Carries a string instead of a member"

except Exception as error:  # the declaration fails; the class name is never bound
    print(type(error).__name__ + ":", error)


## The parentheses are part of the call

`@role_mode` without parentheses hands the class over where the mode belongs. That is refused with its own sentence rather than becoming a later `issubclass()` failure.


In [3]:
try:

    @role_mode
    class ParenthesislessRole(ApplicationRole):
        """Forgot the call."""

        name = "parenthesisless"
        description = "Decorated without parentheses"

except Exception as error:
    print(type(error).__name__ + ":", error)


## A member decides nothing by itself

The decorator checks membership, and stops there. What a mode means belongs to its consumer: `@check_roles` refuses to require a role marked `UNUSED`, warns about `DEPRECATED`, and the access check grants nothing to `SILENCED`.


In [4]:
@role_mode(RoleMode.UNUSED)
class RetiredRole(ApplicationRole):
    """Kept in the code, never required."""

    name = "retired"
    description = "Kept, unused"


print("declared:", RetiredRole._role_mode_info)


try:

    @check_roles(RetiredRole)
    class WithdrawAction:
        """Would have required a role that is not used."""

except RoleSpecTypeError as error:
    print(type(error).__name__ + ":", error)


## What the output says

Every refusal names the role, the value that arrived and all four members — `ALIVE, DEPRECATED, SILENCED, UNUSED` — so the declaration can be fixed from the message alone. The same four lines are shown by the [script](17_a_wrong_role_mode.py), and the invariant is stated on the [reference page](../../docs/reference/intents-and-invariants.md).
